In [4]:
import json

dictio_path = "/lustre/fswork/projects/rech/dki/ujo91el/code/DSI_LIDAR_HD/dictio_results/"
current_dictio = "results_dsi3d_east_global.json"

def load_json(filepath):
    with open(filepath, "r") as f:
        return json.load(f)
        
dsi3d_results = load_json(dictio_path + current_dictio) 


In [10]:
for i in range(len(dsi3d_results)):
    # print("query_idx", dsi3d_results[i]['query_idx'], dsi3d_results[i]['Top10_id'] )
    print("query_idx", dsi3d_results[i]['query_idx'])

query_idx 0
query_idx 1
query_idx 2
query_idx 3
query_idx 4
query_idx 5
query_idx 6
query_idx 7
query_idx 8
query_idx 9
query_idx 10
query_idx 11
query_idx 12
query_idx 13
query_idx 14
query_idx 15
query_idx 16
query_idx 17
query_idx 18
query_idx 19
query_idx 20
query_idx 21
query_idx 22
query_idx 23
query_idx 24
query_idx 25
query_idx 26
query_idx 27
query_idx 28
query_idx 29
query_idx 30
query_idx 31
query_idx 32
query_idx 33
query_idx 34
query_idx 35
query_idx 36
query_idx 37
query_idx 38
query_idx 39
query_idx 40
query_idx 41
query_idx 42
query_idx 43
query_idx 44
query_idx 45
query_idx 46
query_idx 47
query_idx 48
query_idx 49
query_idx 50
query_idx 51
query_idx 52
query_idx 53
query_idx 54
query_idx 55
query_idx 56
query_idx 57
query_idx 58
query_idx 59
query_idx 60
query_idx 61
query_idx 62
query_idx 63
query_idx 64
query_idx 65
query_idx 66
query_idx 67
query_idx 68
query_idx 69
query_idx 70
query_idx 71
query_idx 72
query_idx 73
query_idx 74
query_idx 75
query_idx 76
query_idx

In [11]:
# D3feat

In [5]:
from utils.config import Config
config = Config()

In [10]:
d3feat_model_path = "/lustre/fswork/projects/rech/dki/ujo91el/SpectralGV_D3Feat/models/D3Feat"
config.load(d3feat_model_path)

################################
/lustre/fswork/projects/rech/dki/ujo91el/SpectralGV_D3Feat/models/D3Feat


In [17]:
from datasets.common import Dataset 

class MiniDataset(Dataset):
    """
    A class to create and manage a mini point cloud dataset.
    """

    def __init__(self, files, retranche, voxel_size=0.03):
        """
        Initializes the MiniDataset by loading and splitting point clouds.

        Args:
            files (list): List of paths to point cloud files.
            retranche (int): Number of points to extract per iteration.
            voxel_size (float, optional): Voxel size for downsampling. Defaults to 0.03.

        Attributes:
            anc_points (dict): Dictionary containing split point cloud data.
            ids_list (dict): Dictionary mapping split data to filenames.
            num_test (int): Total number of subsets created.
        """
        Dataset.__init__(self, 'Mini')
        self.num_test = 0
        self.anc_points = {"train": [], "test": []}
        self.ids_list = {"train": [], "test": []}

        for filename in files:
            pcd = o3d.io.read_point_cloud(filename)
            pcd = pcd.voxel_down_sample(voxel_size=voxel_size)
            points = np.array(pcd.points)
            for i in range(0, len(points), retranche):
                print(f"point: {len(points[i : i+retranche])} filename: {filename}")
                self.anc_points['test'] += [points[i : i+retranche]]
                self.ids_list['test'] += [filename]
                self.num_test += 1

    def get_batch_gen(self, split, config):
        """
        Generates a data batch for training or testing.

        Args:
            split (str): Dataset split ('train' or 'test').
            config (dict): Configuration for the generator.

        Returns:
            tuple: (random_balanced_gen, gen_types, gen_shapes)
                - random_balanced_gen (generator): Yields batches of data.
                - gen_types (tuple): Data types for the generator output.
                - gen_shapes (tuple): Shapes for the generator output.
        """
        def random_balanced_gen():
            gen_indices = np.arange(self.num_test)
            for p_i in gen_indices:
                anc_id = self.ids_list['test'][p_i]
                pos_id = self.ids_list['test'][p_i]
                anc_points = self.anc_points['test'][p_i].astype(np.float32)
                pos_points = self.anc_points['test'][p_i].astype(np.float32)
                anc_keypts = np.array([])
                pos_keypts = np.array([])
                yield (np.concatenate([anc_points, pos_points], axis=0),
                       anc_keypts, pos_keypts,
                       np.array([p_i, p_i], dtype=np.int32),
                       np.array([anc_points.shape[0], pos_points.shape[0]]),
                       np.array([anc_id, pos_id]),
                       np.concatenate([anc_points, pos_points], axis=0))

        gen_types = (tf.float32, tf.int32, tf.int32, tf.int32, tf.int32, tf.string, tf.float32)
        gen_shapes = ([None, 3], [None], [None], [None], [None], [None], [None, 3])
        return random_balanced_gen, gen_types, gen_shapes

    def get_tf_mapping(self, config):
        """
        Returns a TensorFlow mapping function to transform raw data into model inputs.

        Args:
            config (dict): Configuration for the mapping.

        Returns:
            function: A tf_map function that processes raw data into formatted inputs.
        """
        def tf_map(anc_points, anc_keypts, pos_keypts, obj_inds, stack_lengths, ply_id, backup_points):
            batch_inds = self.tf_get_batch_inds(stack_lengths)
            stacked_features = tf.ones((tf.shape(anc_points)[0], 1), dtype=tf.float32)
            anchor_input_list = self.tf_descriptor_input(config, anc_points, stacked_features, stack_lengths, batch_inds)
            return anchor_input_list + [stack_lengths, anc_keypts, pos_keypts, ply_id, backup_points]

        return tf_map

ImportError: Traceback (most recent call last):
  File "/gpfslocalsup/pub/anaconda-py3/2019.10/envs/tensorflow-gpu-1.13.2/lib/python3.7/site-packages/tensorflow/python/pywrap_tensorflow.py", line 58, in <module>
    from tensorflow.python.pywrap_tensorflow_internal import *
  File "/gpfslocalsup/pub/anaconda-py3/2019.10/envs/tensorflow-gpu-1.13.2/lib/python3.7/site-packages/tensorflow/python/pywrap_tensorflow_internal.py", line 28, in <module>
    _pywrap_tensorflow_internal = swig_import_helper()
  File "/gpfslocalsup/pub/anaconda-py3/2019.10/envs/tensorflow-gpu-1.13.2/lib/python3.7/site-packages/tensorflow/python/pywrap_tensorflow_internal.py", line 24, in swig_import_helper
    _mod = imp.load_module('_pywrap_tensorflow_internal', fp, pathname, description)
  File "/gpfslocalsup/pub/anaconda-py3/2019.10/envs/tensorflow-gpu-1.13.2/lib/python3.7/imp.py", line 242, in load_module
    return load_dynamic(name, filename, file)
  File "/gpfslocalsup/pub/anaconda-py3/2019.10/envs/tensorflo

In [16]:
retranche = 10000000
['/lustre/fsn1/worksf/projects/rech/dki/ujo91el/datas/datasets/sequences/00/ply/000000.ply']
name_point_cloud_files = dataset = MiniDataset(files=name_point_cloud_files, retranche=retranche, voxel_size=0.1)
dataset.init_test_input_pipeline(config)

NameError: name 'MiniDataset' is not defined

In [15]:
import glob
import argparse

import numpy as np
import os
import copy
import sys
import time
from utils.config import Config
from datasets.common import Dataset
from models.KPFCNN_model import KernelPointFCNN

config = Config()
config.load(self.model_path)
model = KernelPointFCNN(dataset.flat_inputs, config)


ImportError: Traceback (most recent call last):
  File "/gpfslocalsup/pub/anaconda-py3/2019.10/envs/tensorflow-gpu-1.13.2/lib/python3.7/site-packages/tensorflow/python/pywrap_tensorflow.py", line 58, in <module>
    from tensorflow.python.pywrap_tensorflow_internal import *
  File "/gpfslocalsup/pub/anaconda-py3/2019.10/envs/tensorflow-gpu-1.13.2/lib/python3.7/site-packages/tensorflow/python/pywrap_tensorflow_internal.py", line 28, in <module>
    _pywrap_tensorflow_internal = swig_import_helper()
  File "/gpfslocalsup/pub/anaconda-py3/2019.10/envs/tensorflow-gpu-1.13.2/lib/python3.7/site-packages/tensorflow/python/pywrap_tensorflow_internal.py", line 24, in swig_import_helper
    _mod = imp.load_module('_pywrap_tensorflow_internal', fp, pathname, description)
  File "/gpfslocalsup/pub/anaconda-py3/2019.10/envs/tensorflow-gpu-1.13.2/lib/python3.7/imp.py", line 242, in load_module
    return load_dynamic(name, filename, file)
  File "/gpfslocalsup/pub/anaconda-py3/2019.10/envs/tensorflo

In [12]:
# Spectral GV